In [1]:
! pip install wordsegment

import numpy as np
import pandas as pd
import nltk, re, random, textwrap
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from wordsegment import load, segment
from sklearn.metrics import accuracy_score

# Download datasets required for Lemmatization and POS tagging
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
load()

# --- 1. HELPER TO CONVERT NLTK TAGS TO WORDNET TAGS ---
def get_wordnet_pos(nltk_tag):
    """
    Lemmatization works best when given a Part-of-Speech (POS) tag.
    This helper maps default NLTK POS tags to WordNet-compatible tags.
    """
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default fallback

# --- 2. DEFINE THE LEMMATIZATION CLEANING FUNCTION ---
def lemmatize_text(text, lemmatizer):
    """
    Tokenizes, POS tags, and lemmatizes a string of text.
    """
    # Tokenize text into individual words
    tokens = nltk.word_tokenize(text.lower())

    # Get POS tags for the words (e.g., [('running', 'VBG')])
    pos_tags = nltk.pos_tag(tokens)

    # Lemmatize each word using its context-driven POS tag
    clean_tokens = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags
    ]

    # Recombine back into a single clean string
    return " ".join(clean_tokens)


# Function for fixing the NaN values of 'negativereason' column in data set
def fix_negativereason(record):
  if record['negativereason'] != record['negativereason']:
    if record['airline_sentiment'].lower() == 'neutral':
      return 'neutral'
    elif record['airline_sentiment'].lower() == 'positive':
      return 'positive'
    else:
      return 'negative'
  else:
    return record['negativereason']

# Function for fixing the NaN values of 'negativereason_confidence' column in data set
def fix_negativereason_confidence(record):
  if record['negativereason_confidence'] != record['negativereason_confidence']:
    if record['airline_sentiment_confidence'] < 0.50:
      return 0.00
    else:
      return record['airline_sentiment_confidence']
  else:
    return record['negativereason_confidence']

# Encode string sentiment to numeric value
# 1 -> positive, 2-> neutral, 3 -> negative
def encode_sentiment(record):
  if record['airline_sentiment'].lower() == 'positive':
    return 1
  elif record['airline_sentiment'].lower() == 'neutral':
    return 2
  elif record['airline_sentiment'].lower() == 'negative':
    return 3
  else:
    return -1

def remove_special_chars(record):
  text = re.sub(r"^@", "", record['text'])
  text = re.sub(r"[^a-zA-Z0-9 ]", "", text)
  return text

def remove_stop_words(text, stop_words):
  tokens = list(dict.fromkeys(word_tokenize(text.lower())))
  fixed_words = [ " ".join(segment(token)) for token in tokens]
  fixed_text = " ".join(fixed_words)
  return fixed_text
  #tokens = list(dict.fromkeys(word_tokenize(fixed_text)))
  #cleaned_tokens = [token for token in tokens if token.lower() not in stop_words]
  #return " ".join(cleaned_tokens)

def get_sentiment(value):
  if value == 1:
    return 'Positive'
  elif value == 2:
    return 'Neutral'
  elif value == 3:
    return 'Negative'
  else:
    return "Unknown"

# Step 1 : Read the data set

# Reading training data
tweets_train_df = pd.read_csv("/content/sample_data/Tweets-train.csv")

# Reading test data
tweets_test_df = pd.read_csv("/content/sample_data/Tweets-test.csv")

combined_tweets_df = pd.concat([tweets_train_df, tweets_test_df], ignore_index=True)

feature_columns = [
  'airline_sentiment',
  'airline_sentiment_confidence',
  'negativereason',
  'negativereason_confidence',
  'text'
]

# Selecting columns which are required for sentiment analysis
cleaned_df = combined_tweets_df[feature_columns].copy()

# Select rows which are realistic
cleaned_df = cleaned_df[
    (cleaned_df['airline_sentiment_confidence'] > 0.50) |
    (cleaned_df['negativereason_confidence'] > 0.50) |
    (cleaned_df['negativereason'].notna())
]

# Cleaning trial 1
cleaned_df['negativereason'] = cleaned_df.apply(fix_negativereason, axis=1)

# Cleaning trial 2
cleaned_df['negativereason_confidence'] = cleaned_df.apply(fix_negativereason_confidence, axis=1)

# Cleaning trial 3
cleaned_df = cleaned_df[cleaned_df['negativereason_confidence'] >= 0.50]

# Cleaning trial 4
cleaned_df['text'] = cleaned_df.apply(remove_special_chars, axis=1)

# Cleaning trial 5
cleaned_df['airline_sentiment'] = cleaned_df.apply(encode_sentiment, axis=1)

training_set_corpus = list(cleaned_df['text'])
stop_words = set(stopwords.words('english'))
# Initialize the Lemmatizer
lemmatizer = WordNetLemmatizer()
training_set_corpus = [ lemmatize_text(remove_stop_words(text, stop_words), lemmatizer) for text in training_set_corpus]

y = cleaned_df['airline_sentiment']
# Split the data into data set for training and data set for testing. In below code we are using 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(training_set_corpus, y, test_size=0.2, random_state=42)


# Create vectors
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    max_df=1.0,       # Don't drop frequent words
    min_df=1,         # Don't drop rare words
    max_features=None,# Don't cap the vocabulary size
    sublinear_tf=True # Use log scaling to prevent dilution
)

X = tfidf.fit_transform(X_train)

model = LogisticRegression()
model.fit(X, y_train)

number_of_tests = int(input("Enter number of tests: "))
successfull_tests = 0
for i in range(number_of_tests):
  index = random.randint(1, len(X_test))
  test_tweet = X_test[index:index + 1][0]
  tweet_sentiment = get_sentiment(y_test[index: index + 1].iloc[0])
  unseen_text = [ lemmatize_text(remove_stop_words(text, stop_words), lemmatizer) for text in [test_tweet]]
  unseen_vector = tfidf.transform(unseen_text)
  prediction = model.predict(unseen_vector)
  predicted_tweet_sentiment = get_sentiment(prediction[0])
  if tweet_sentiment == predicted_tweet_sentiment:
    successfull_tests = successfull_tests + 1
  #print(textwrap.shorten(test_tweet, width=30, placeholder="..."), " : ", tweet_sentiment, ", Predicted:", predicted_tweet_sentiment, "\n")

unseen_vectors = tfidf.transform(X_test)
prediction = model.predict(unseen_vectors)

train_accuracy = model.score(X, y_train)
test_accuracy = model.score(unseen_vectors, y_test)
accuracy = accuracy_score(y_test, prediction)

model_matrix = {
    'Training Accuracy': [f'{train_accuracy * 100:.2f}%'],
    'Test Accuracy': [f'{test_accuracy * 100:.2f}%'],
    'Model Accuracy': [f'{accuracy * 100:.2f}%'],
    'Random Test Count': [number_of_tests],
    'Random Test Success %': [f'{(successfull_tests/number_of_tests) * 100:.2f}%']
}

model_matrix_df = pd.DataFrame(model_matrix)
display(model_matrix_df)
print("\n")

while(1):
  print("Enter 'Exit' to quit.")
  test_string = input("Enter a tweet to analyze: ")
  test_string = test_string.rstrip("\n")
  if test_string.lower() == 'exit':
    break
  unseen_text = [ lemmatize_text(remove_stop_words(text, stop_words), lemmatizer) for text in [test_string]]
  unseen_vector = tfidf.transform(unseen_text)

  # Get the raw probability scores to see how confident the model is
  probabilities = model.predict_proba(unseen_vector)[0]
  predicted_tweet_sentiment = get_sentiment(prediction[0])

  print(f'Processed Text: "{unseen_text}"')
  print(f'Predicted: {predicted_tweet_sentiment}')
  print(f'Confidence - Pos: {probabilities[0]:.2f}, Neu: {probabilities[1]:.2f}, Neg: {probabilities[2]:.2f}\n')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 44.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Enter number of tests: 100


,Training Accuracy,Test Accuracy,Model Accuracy,Random Test Count,Random Test Success %
0,92.82%,81.59%,81.59%,100,78.00%




Enter 'Exit' to quit.
Enter a tweet to analyze: My travel with USAirways was very good
Processed Text: "['my travel with usairways be very good']"
Predicted: Negative
Confidence - Pos: 0.26, Neu: 0.14, Neg: 0.61

Enter 'Exit' to quit.
Enter a tweet to analyze: my travel with usairways was awesome
Processed Text: "['my travel with usairways be awesome']"
Predicted: Negative
Confidence - Pos: 0.40, Neu: 0.20, Neg: 0.40

Enter 'Exit' to quit.
Enter a tweet to analyze: my travel with usairways was incredible
Processed Text: "['my travel with usairways be incredible']"
Predicted: Negative
Confidence - Pos: 0.15, Neu: 0.23, Neg: 0.62

Enter 'Exit' to quit.
Enter a tweet to analyze: Exit
